# 10 · Rotation-gated 물리 Decoder — 최종 모델 (EXP #38, LB 0.6996 ⭐)

직전 base = rnn_solo(LB 0.6920), plateau ~0.692. cascade·RNN은 전부 +0.000X sub-std로 정체.

**가정**: 정체의 원인은 우리 모든 모델이 **CV(등속) 직선 base를 공유**해 곡률(minority, acc≥5)을 구조적으로 못 휘는 데 있다. #36 결론대로 "OOF→LB 전이되는 저상관 새 물리"가 필요하고, 그 후보가 **회전(곡선 외삽)**이다. 단 고정 회전(해석적 CTRV)은 후속 모델에 흡수돼 죽으니, **NN이 감쇠를 학습한 회전 + 저속·직진 차단 gate**여야 한다(외부 0.699 `HyperPhysics_xy2` 이식).

**실험**: 단일 MLP가 좌표를 직접 회귀하지 않고 **물리 식의 계수만** 학습 — local frame R 위에서 `pred = p_last + R·[w_v·e^(−exp_v)·rodrigues(v_EMA, ω) + w_a·e^(−exp_a)·a_local]`, ω=(ω_hist 해석적 + ω_delta 학습)·gate. soft-hit + MSE + NLL loss, 내부 sliding-window 증강 + θ 가중 oversampling. honest 5-fold(stratify=minority) × 3 seed **solo**, val/test는 real 11점. G3 minority hard-stop은 신 아키텍처라 의도적 override하고 **LB로 직접 판정**.

**결과**: **LB 0.6996 (+0.0076) — 신 SOTA, plateau 돌파, 1등 0.70까지 −0.0004**. OOF 0.6765인데 LB lift +0.0231(rnn_solo +0.0178보다 큼) = **물리 제약이 과적합을 줄여 generalization 우위**("제약 없는 NN이 진 것이지 NN이 진 게 아니다", #25 반례). 이 rotgate가 최종 채택 base이며, 대회는 **최종 LB 0.7002 · 12등**으로 종료.

## 셀 0 — imports + 상수 + GPU

In [ ]:
import os, time, json, math, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold

DT, DT_PRED = 0.04, 0.08
N_FOLDS = 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01
SEEDS = [42, 7, 123]

# 학습 recipe
BATCH = 256
EPOCHS = 12
PATIENCE = 5
GRAD_CLIP = 1.0
SCHED_STEP, SCHED_GAMMA = 3, 0.5

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device = {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  torch {torch.__version__}')
os.makedirs('results', exist_ok=True)
print('imports + 상수 OK')

## 셀 1 — helper (physics base + hit_rate + splits + seed)

In [ ]:
def physics_baseline(traj):
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return (p0 + v_last * DT_PRED).astype(np.float32)


def hit_rate(pred, label, r=HIT_RADIUS):
    return float((np.linalg.norm(pred - label, axis=1) < r).mean())


def hr_triple(hit_arr, minority_mask):
    return dict(overall=float(hit_arr.mean()),
                major=float(hit_arr[~minority_mask].mean()),
                minor=float(hit_arr[minority_mask].mean()))


def make_splits(stratify_int, seed, n_folds=N_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.arange(len(stratify_int)), stratify_int)]


def set_seed(seed):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print('helper 정의 완료')

## 셀 2 — Drive mount + 데이터 (traj_train/test, y_train)

In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'
DRIVE_BASE = '/content/drive/MyDrive'

need_drive = not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR) and os.path.exists(CACHE_TRAJ_TS))
if need_drive:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    # cache 우선 Drive 복사 시도
    for p in [CACHE_TRAJ_TR, CACHE_Y_TR, CACHE_TRAJ_TS]:
        src = f'{DRIVE_BASE}/{p}'
        if not os.path.exists(p) and os.path.exists(src):
            shutil.copy(src, p)
    if not os.path.exists('/content/open') and not os.path.exists(CACHE_TRAJ_TR):
        !unzip -q -o "{DRIVE_BASE}/open.zip" -d /content/

def _resolve_data_dir():
    for cand in ['/content/open', '/content']:
        if os.path.exists(f'{cand}/train_labels.csv'):
            return cand
    return None

def _resolve_sample_sub(data_dir):
    for p in [f'{data_dir}/sample_submission.csv', f'{data_dir}/submission/sample_submission.csv']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'sample_submission.csv not found under {data_dir}')

DATA_DIR = _resolve_data_dir()
if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR).astype(np.float32); y_train = np.load(CACHE_Y_TR).astype(np.float32)
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        traj_train[i] = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train); np.save(CACHE_Y_TR, y_train)

sample_sub = pd.read_csv(_resolve_sample_sub(DATA_DIR)) if DATA_DIR else None
if sample_sub is not None:
    test_ids = sample_sub['id'].tolist()
else:
    test_ids = [f'TEST_{i:05d}' for i in range(10000)]
if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS).astype(np.float32)
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        traj_test[i] = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)

assert traj_train.shape == (10000, 11, 3) and traj_test.shape == (10000, 11, 3)
print(f'train {traj_train.shape}, test {traj_test.shape} OK')

## 셀 3 — physics CV base + minority mask + sanity

In [ ]:
base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
v = (traj_train[:, 1:] - traj_train[:, :-1]) / DT
a = (v[:, 1:] - v[:, :-1]) / DT
minority_mask_tr = np.linalg.norm(a[:, -1], axis=1) >= MINORITY_THRESHOLD
vt = (traj_test[:, 1:] - traj_test[:, :-1]) / DT
at = (vt[:, 1:] - vt[:, :-1]) / DT
minority_mask_ts = np.linalg.norm(at[:, -1], axis=1) >= MINORITY_THRESHOLD
print(f'base CV HR (sanity): {hit_rate(base_train, y_train):.4f}')
print(f'minority train {minority_mask_tr.sum()}/{len(minority_mask_tr)} ({minority_mask_tr.mean()*100:.1f}%)')

## 셀 4 — 데이터셋 + feature 추출 + 빌딩 블록

SlidingWindowDataset(등속외삽 패딩 + extended target + θ 가중) + extract_features(24-dim local feature)
+ ResBlock / PriorBiasedLinear(prior 초기화 선형층) + rodrigues_rotate(축-각 회전).

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, X, y, min_win=3, mode='extended', device='cpu'):
        X_t = torch.tensor(X, dtype=torch.float32)
        y_t = torch.tensor(y, dtype=torch.float32)
        windows = []
        for i in range(len(X)):
            targets = [4,5,6,7,8,9,10,12] if mode == 'extended' else [12, 10]
            for target_idx in targets:
                end_idx = target_idx - 2
                max_w = end_idx + 2 if mode == 'extended' else (12 if target_idx == 12 else 10)
                for w in range(min_win, max_w):
                    windows.append((i, w, target_idx))
        X_list, y_list = [], []
        for i, w, target_idx in windows:
            X_orig = X_t[i]; end_idx = target_idx - 2
            pts = X_orig[end_idx - w + 1: end_idx + 1]
            target = y_t[i] if target_idx == 12 else X_orig[target_idx]
            if w < 11:
                v0 = pts[1] - pts[0]; n_pad = 11 - w
                js = torch.arange(n_pad, 0, -1, dtype=torch.float32)
                pad = pts[0:1] - js.unsqueeze(1) * v0.unsqueeze(0)
                X_padded = torch.cat([pad, pts], dim=0)
            else:
                X_padded = pts.clone()
            X_list.append(X_padded); y_list.append(target)
        self.X_all = torch.stack(X_list).to(device)
        self.y_all = torch.stack(y_list).to(device)
        diffs = self.X_all[:, 1:] - self.X_all[:, :-1]
        n1 = diffs[:, 1:].norm(dim=2).clamp(min=1e-8)
        n2 = diffs[:, :-1].norm(dim=2).clamp(min=1e-8)
        cos_t = ((diffs[:, 1:] * diffs[:, :-1]).sum(dim=2) / (n1 * n2)).clamp(-1, 1)
        theta_last = torch.acos(cos_t[:, -1])
        self.theta_weights = (1.0 + 4.0 * (theta_last / 1.0).clamp(0, 1)).cpu().numpy()
    def __len__(self):
        return len(self.X_all)
    def __getitem__(self, idx):
        return self.X_all[idx], self.y_all[idx]


def _ema_va_local(diffs_local, alpha, beta):
    B, T, _ = diffs_local.shape
    one_m_a = 1.0 - alpha; one_m_b = 1.0 - beta
    vs = diffs_local.new_empty(B, T, 3)
    vv = diffs_local[:, 0]; vs[:, 0] = vv
    for t in range(1, T):
        vv = alpha * diffs_local[:, t] + one_m_a * vv; vs[:, t] = vv
    vl = vs[:, -1]
    ad = vs[:, 1:] - vs[:, :-1]; aa = ad[:, 0]
    for t in range(1, T - 1):
        aa = beta * ad[:, t] + one_m_b * aa
    return vl, aa


def _soft_hit_loss(pred, target, thr=0.013012, k=408.348):
    return (1 - torch.sigmoid(-(torch.norm(pred - target, dim=1) - thr) * k)).mean()


def extract_features(X, mean_stats=None, std_stats=None, dir_net=None, heading_mode='3step'):
    device = X.device
    p_last = X[:, 10]
    diffs = X[:, 1:] - X[:, :-1]
    n1 = diffs[:, 1:].norm(dim=2, keepdim=True) + 1e-8
    n2 = diffs[:, :-1].norm(dim=2, keepdim=True) + 1e-8
    cos_t = ((diffs[:, 1:] * diffs[:, :-1]).sum(dim=2, keepdim=True) / (n1 * n2)).clamp(-1, 1)
    theta_seq = torch.acos(cos_t).squeeze(2)
    theta = theta_seq[:, -1:]
    theta_mean = theta_seq.mean(1, keepdim=True)
    theta_std = theta_seq.std(1, keepdim=True)
    theta_vel = theta_seq[:, -1:] - theta_seq[:, -2:-1]
    theta_acc = theta_seq[:, -1:] - 2 * theta_seq[:, -2:-1] + theta_seq[:, -3:-2]
    theta_trend = theta_seq[:, -1:] - theta_seq[:, -3:].mean(1, keepdim=True)
    if dir_net is not None:
        speed_seq = diffs.norm(dim=2)
        state = torch.cat([speed_seq, theta_seq], dim=1)
        if dir_net[0].in_features == 29:
            z_speed_seq = diffs[:, :, 2].abs()
            state = torch.cat([state, z_speed_seq], dim=1)
        weights = F.softmax(dir_net(state), dim=1)
        v_sm = (diffs * weights.unsqueeze(2)).sum(dim=1)
    else:
        if heading_mode == '3step':
            v_sm = (3 * diffs[:, -1] + 2 * diffs[:, -2] + diffs[:, -3]) / 6.0
        else:
            v_sm = diffs[:, -1]
    fwd = v_sm / (v_sm.norm(dim=1, keepdim=True) + 1e-8)
    up_w = torch.zeros_like(fwd); up_w[:, 2] = 1.0
    up_w[fwd[:, 2].abs() > 0.99] = torch.tensor([0., 1., 0.], device=device)
    right = torch.cross(fwd, up_w, dim=1)
    right = right / (right.norm(dim=1, keepdim=True) + 1e-8)
    up = torch.cross(right, fwd, dim=1)
    up = up / (up.norm(dim=1, keepdim=True) + 1e-8)
    R = torch.stack([fwd, right, up], dim=2)
    v_last = diffs[:, -1]; v_prev1 = diffs[:, -2]
    speed = v_last.norm(dim=1, keepdim=True)
    a_last = v_last - v_prev1
    acc_mag = a_last.norm(dim=1, keepdim=True)
    v_local = torch.matmul(v_last.unsqueeze(1), R).squeeze(1)
    a_local = torch.matmul(a_last.unsqueeze(1), R).squeeze(1)
    X_local = torch.matmul(X - p_last.unsqueeze(1), R)
    p_std_local = X_local.std(1)
    v_local_abs = v_local.abs()
    jerk_g = diffs[:, -1] - 2 * diffs[:, -2] + diffs[:, -3]
    jerk_l = torch.matmul(jerk_g.unsqueeze(1), R).squeeze(1)
    jerk_mag = jerk_g.norm(dim=1, keepdim=True)
    features = torch.cat([v_local, a_local, speed, acc_mag,
        theta, theta_mean, theta_std, theta_trend, theta_vel, theta_acc,
        p_std_local, v_local_abs, jerk_l, jerk_mag], dim=1)
    if mean_stats is None or std_stats is None:
        mean_stats = features.mean(0, keepdim=True)
        std_stats = features.std(0, keepdim=True) + 1e-8
    features_scaled = (features - mean_stats) / std_stats
    return features_scaled, diffs, p_last, theta, theta_mean, theta_std, theta_seq, R, speed, mean_stats, std_stats


class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(),
                                 nn.Dropout(0.15), nn.Linear(dim, dim))
        self.ln = nn.LayerNorm(dim)
    def forward(self, x):
        return self.ln(x + self.net(x))


class PriorBiasedLinear(nn.Module):
    def __init__(self, in_features, out_features, prior_bias):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.register_buffer('prior_bias', prior_bias.clone().detach())
        with torch.no_grad():
            nn.init.zeros_(self.linear.weight); nn.init.zeros_(self.linear.bias)
    def forward(self, x):
        return self.linear(x) + self.prior_bias


def rodrigues_rotate(vv, w):
    theta = w.norm(dim=1, keepdim=True)
    k = w / (theta + 1e-8)
    cos_t = torch.cos(theta); sin_t = torch.sin(theta)
    dot = (vv * k).sum(dim=1, keepdim=True)
    cross = torch.cross(k, vv, dim=1)
    return vv * cos_t + cross * sin_t + k * dot * (1.0 - cos_t)


print('SlidingWindowDataset / extract_features / blocks / rodrigues 정의 완료')

## 셀 5 — HyperPhysics_xy2 (rotation gate + 감쇠 decoder + loss)

In [ ]:
class HyperPhysics_xy2(nn.Module):
    def __init__(self, input_dim=24, **kwargs):
        super().__init__()
        self.sh_thr    = kwargs.pop('sh_thr',    0.013012)
        self.sh_k      = kwargs.pop('sh_k',      408.348044)
        self.mse_w     = kwargs.pop('mse_w',     129.172037)
        self.local_w   = kwargs.pop('local_w',   0.050941)
        self.theta_thr = kwargs.pop('theta_thr', 1.087618)
        self.speed_thr = kwargs.pop('speed_thr', 0.034583)
        self.lr = 0.005400; self.wd = 0.005659
        self.register_buffer('mean_stats', torch.zeros(1, input_dim))
        self.register_buffer('std_stats', torch.ones(1, input_dim))
        prior_dir = torch.tensor([-10.,-10.,-10.,-10.,-10.,-10.,-10., 0., 0.693, 1.098])
        self.dir_net = nn.Sequential(nn.Linear(29, 24), nn.LayerNorm(24), nn.GELU(),
                                     PriorBiasedLinear(24, 10, prior_dir))
        prior_ema = torch.zeros(6)
        self.temporal_net = nn.Sequential(nn.Linear(9, 32), nn.LayerNorm(32), nn.GELU(),
                                          PriorBiasedLinear(32, 6, prior_ema))
        prior_dyn = torch.tensor([0.,0.,0.,0.,0.,0.,
            -4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,
            -4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.,-4.])
        self.dynamics_net = nn.Sequential(nn.Linear(input_dim, 96), nn.LayerNorm(96), nn.GELU(),
                                          ResBlock(96), PriorBiasedLinear(96, 30, prior_dyn))
        self.omega_w = nn.Parameter(torch.tensor([0.0, -0.5, -1.0]))
        self.omega_net = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, 48),
                                       nn.GELU(), nn.Linear(48, 3))
        with torch.no_grad():
            nn.init.normal_(self.omega_net[-1].weight, std=0.01)
            nn.init.zeros_(self.omega_net[-1].bias)
        self.diffusion_net = nn.Sequential(nn.Linear(input_dim, 32), nn.LayerNorm(32), nn.GELU(),
                                           nn.Linear(32, 3))
    def get_features(self, X, mean_stats=None, std_stats=None):
        return extract_features(X, mean_stats, std_stats, self.dir_net, heading_mode='3step')
    def _ema_va_local(self, diffs_local, alpha, beta):
        return _ema_va_local(diffs_local, alpha, beta)
    @staticmethod
    def _rotation_vector(d_prev, d_curr):
        n_prev = d_prev.norm(dim=1, keepdim=True).clamp(min=1e-8)
        n_curr = d_curr.norm(dim=1, keepdim=True).clamp(min=1e-8)
        d_hat_prev = d_prev / n_prev; d_hat_curr = d_curr / n_curr
        cross = torch.linalg.cross(d_hat_prev, d_hat_curr, dim=1)
        sin_t = cross.norm(dim=1, keepdim=True).clamp(min=1e-8)
        cos_t = (d_hat_prev * d_hat_curr).sum(1, keepdim=True).clamp(-0.9999, 0.9999)
        theta = torch.atan2(sin_t, cos_t)
        speed_gate = torch.sigmoid((n_prev + n_curr) * 500 - 5)
        return cross / sin_t * theta * speed_gate
    def forward(self, features, diffs, p_last, theta, speed, R):
        B = diffs.shape[0]
        ema_raw = self.temporal_net(features[:, 8:17])
        alpha = torch.sigmoid(ema_raw[:, 0:3]) * 0.8 + 0.1
        beta  = torch.sigmoid(ema_raw[:, 3:6]) * 0.199 + 0.8
        dyn_raw = self.dynamics_net(features)
        w_v = 2.0 + dyn_raw[:, 0:3]; w_a = 1.0 + dyn_raw[:, 3:6]
        v_local_abs = features[:, 17:20]; v_local_abs2 = v_local_abs * v_local_abs
        theta2 = theta * theta
        exp_v = (F.softplus(dyn_raw[:, 6:9]) * v_local_abs + F.softplus(dyn_raw[:, 9:12]) * v_local_abs2 +
                 F.softplus(dyn_raw[:, 12:15]) * theta + F.softplus(dyn_raw[:, 15:18]) * theta2)
        exp_a = (F.softplus(dyn_raw[:, 18:21]) * v_local_abs + F.softplus(dyn_raw[:, 21:24]) * v_local_abs2 +
                 F.softplus(dyn_raw[:, 24:27]) * theta + F.softplus(dyn_raw[:, 27:30]) * theta2)
        diffs_local = torch.matmul(diffs, R)
        vl, al = self._ema_va_local(diffs_local, alpha, beta)
        diff_speed = diffs_local.norm(dim=2)
        def rv_masked(ka, kb):
            rv = self._rotation_vector(diffs_local[:, ka], diffs_local[:, kb])
            valid = ((diff_speed[:, ka] > 1e-5) & (diff_speed[:, kb] > 1e-5)).float()
            return rv * valid.unsqueeze(1), valid
        ov1, vm1 = rv_masked(-2, -1); ov2, vm2 = rv_masked(-3, -2); ov3, vm3 = rv_masked(-4, -3)
        w_logits = self.omega_w.view(1, 3).expand(B, -1)
        masks = torch.stack([vm1, vm2, vm3], dim=1)
        w_logits_masked = w_logits.masked_fill(masks == 0, -1e9)
        w_attn = F.softmax(w_logits_masked, dim=1)
        omega_hist = (w_attn[:, 0].unsqueeze(1) * ov1 + w_attn[:, 1].unsqueeze(1) * ov2 +
                      w_attn[:, 2].unsqueeze(1) * ov3)
        current_speed = speed.view(B, 1) if speed is not None else diff_speed[:, -1].unsqueeze(1)
        omega_speed_gate = torch.sigmoid(current_speed * 500 - 5)
        omega_delta = self.omega_net(features) * omega_speed_gate
        theta_scalar = theta.view(B, 1)
        theta_gate = torch.sigmoid((theta_scalar - self.theta_thr) * 10)
        speed_gate_strong = torch.sigmoid((current_speed - self.speed_thr) * 200)
        rotation_gate = theta_gate * speed_gate_strong
        omega = (omega_hist + omega_delta) * rotation_gate
        v_rotated = rodrigues_rotate(vl, omega)
        pred_local = (w_v * torch.exp(-exp_v)) * v_rotated + (w_a * torch.exp(-exp_a)) * al
        log_var = self.diffusion_net(features).clamp(min=-5.0, max=5.0)
        pred_global = p_last + torch.einsum('nij,nj->ni', R, pred_local)
        return pred_global, pred_local, log_var
    def compute_loss(self, pp, yr, pred_local=None, yr_local=None, log_var=None, p_last=None, theta=None, **kwargs):
        sh = _soft_hit_loss(pp, yr, thr=self.sh_thr, k=self.sh_k)
        loss = sh + self.mse_w * F.mse_loss(pp, yr)
        if pred_local is not None and yr_local is not None and log_var is not None:
            squared_error = (pred_local - yr_local) ** 2
            nll_loss = 0.5 * (torch.exp(-log_var) * squared_error + log_var)
            loss = loss + self.local_w * nll_loss.mean()
        return loss


_m = HyperPhysics_xy2(); print(f'HyperPhysics_xy2 params: {sum(p.numel() for p in _m.parameters()):,}'); del _m

## 셀 6 — train_one_fold (증강 train → real 11점 val/test)

train = train row의 SlidingWindowDataset(extended + θ 가중 sampler). val/test = real 11점 직접 입력.
mean/std는 fold train에서 1회 고정. val R-hit 기준 early-stop, best state 복원.

In [ ]:
@torch.no_grad()
def predict_global(model, X_np):
    Xt = torch.tensor(X_np, dtype=torch.float32, device=DEVICE)
    out = []
    for i in range(0, len(Xt), 4096):
        xb = Xt[i:i+4096]
        ft, df, pl, th, _, _, _, Rt, sp, _, _ = model.get_features(xb, model.mean_stats, model.std_stats)
        pp, _, _ = model(ft, df, pl, th, sp, Rt)
        out.append(pp.cpu().numpy())
    return np.concatenate(out).astype(np.float32)


def train_one_fold(X_tr, y_tr, X_va, y_va, X_ts, model_seed):
    set_seed(model_seed)
    train_ds = SlidingWindowDataset(X_tr, y_tr, min_win=3, mode='extended', device=DEVICE)
    sampler = WeightedRandomSampler(train_ds.theta_weights, len(train_ds), replacement=True)
    loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler)
    model = HyperPhysics_xy2().to(DEVICE)
    with torch.no_grad():
        _, _, _, _, _, _, _, _, _, mn, st = model.get_features(
            torch.tensor(X_tr, dtype=torch.float32, device=DEVICE))
        model.mean_stats.copy_(mn); model.std_stats.copy_(st)
    opt = torch.optim.AdamW(model.parameters(), lr=model.lr, weight_decay=model.wd)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=SCHED_STEP, gamma=SCHED_GAMMA)
    best_rhit, best_state, pc, last_ep = -1.0, None, 0, 0
    for ep in range(EPOCHS):
        model.train()
        for Xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            ft, df, pl, th, _, _, _, Rt, sp, _, _ = model.get_features(Xb, model.mean_stats, model.std_stats)
            pp, pred_local, log_var = model(ft, df, pl, th, sp, Rt)
            yr_local = torch.matmul((yb - pl).unsqueeze(1), Rt).squeeze(1)
            loss = model.compute_loss(pp, yb, pred_local, yr_local, log_var, p_last=pl, theta=th)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        sched.step()
        model.eval()
        pg = predict_global(model, X_va)
        rhit = float((np.linalg.norm(pg - y_va, axis=1) < HIT_RADIUS).mean())
        if rhit > best_rhit:
            best_rhit = rhit
            best_state = {kk: vv.detach().clone() for kk, vv in model.state_dict().items()}
            pc = 0
        else:
            pc += 1
            if pc >= PATIENCE:
                break
        last_ep = ep
    model.load_state_dict(best_state); model.eval()
    oof = predict_global(model, X_va)
    test = predict_global(model, X_ts)
    return oof, test, best_rhit, last_ep + 1

print('train_one_fold 정의 완료')

## 셀 7 — 5-fold OOF (seed 42 / 7 / 123)

각 seed마다 StratifiedKFold(stratify=minority)로 5-fold 학습. fold별 OOF(held-out row)와 test pred(fold 평균)를 누적, seed별 `results/rotgate_e38_{oof,test}_seed*.npy` 저장.

In [ ]:
def run_seed(seed):
    folds = make_splits(minority_mask_tr.astype(int), seed=seed)
    oof = np.full((10000, 3), np.nan, dtype=np.float32)
    test = np.zeros((10000, 3), dtype=np.float32)
    rhits, eps_used = [], []
    t0 = time.time()
    for fi, (tr_idx, va_idx) in enumerate(folds):
        oof_f, test_f, rhit, n_ep = train_one_fold(
            traj_train[tr_idx], y_train[tr_idx], traj_train[va_idx], y_train[va_idx],
            traj_test, model_seed=seed * 1000 + fi)
        oof[va_idx] = oof_f
        test += test_f / N_FOLDS
        rhits.append(rhit); eps_used.append(n_ep)
        print(f'    seed={seed} fold={fi}: val_rhit={rhit:.4f} ep={n_ep} ({time.time()-t0:.0f}s cum)')
    HR = hr_triple(np.linalg.norm(oof - y_train, axis=1) < HIT_RADIUS, minority_mask_tr)
    np.save(f'results/rotgate_e38_oof_seed{seed}.npy', oof)
    np.save(f'results/rotgate_e38_test_seed{seed}.npy', test)
    print(f'  seed={seed} DONE: OOF overall={HR["overall"]:.4f} major={HR["major"]:.4f} minor={HR["minor"]:.4f} mean_ep={np.mean(eps_used):.0f}')
    return oof, test, HR

oof_per_seed, test_per_seed, HR_per_seed = {}, {}, {}
for s in SEEDS:
    print(f'=== seed {s} ===')
    o, t, h = run_seed(s)
    oof_per_seed[s] = o; test_per_seed[s] = t; HR_per_seed[s] = h

## 셀 8 — 최종 예측 (3-seed × 5-fold 평균)

3개 seed의 OOF/test를 평균해 최종 예측 산출. overall/major/minor hit-rate 집계.

In [ ]:
seeds_done = list(oof_per_seed.keys())
rot_oof = np.mean([oof_per_seed[s] for s in seeds_done], axis=0).astype(np.float32)
rot_test = np.mean([test_per_seed[s] for s in seeds_done], axis=0).astype(np.float32)
HR_rot = hr_triple(np.linalg.norm(rot_oof - y_train, axis=1) < HIT_RADIUS, minority_mask_tr)
print(f'{len(seeds_done)}-seed OOF mean: '
      f'overall={HR_rot["overall"]:.4f} major={HR_rot["major"]:.4f} minor={HR_rot["minor"]:.4f}')

## 셀 9 — submission CSV 생성 (5-fold × seed mean)

3-seed × 5-fold 평균 test pred로 `submission_rotgate_e38.csv` 생성.

In [ ]:
pred_test = rot_test.astype(np.float32)
assert pred_test.shape == (10000, 3) and np.isfinite(pred_test).all()
sub_path = 'submission_rotgate_e38.csv'
pd.DataFrame({'id': test_ids, 'x': pred_test[:,0], 'y': pred_test[:,1], 'z': pred_test[:,2]}).to_csv(sub_path, index=False)
print(f'submission: {sub_path}')
print(f'  x[{pred_test[:,0].min():.2f},{pred_test[:,0].max():.2f}] '
      f'y[{pred_test[:,1].min():.2f},{pred_test[:,1].max():.2f}] z[{pred_test[:,2].min():.2f},{pred_test[:,2].max():.2f}]')

## 셀 10 — Meta 저장 + Drive 복사 + 다운로드

In [ ]:
def deep_safe(d):
    if isinstance(d, dict): return {k: deep_safe(v) for k, v in d.items()}
    if isinstance(d, (list, tuple)): return [deep_safe(x) for x in d]
    if isinstance(d, np.floating): return float(d)
    if isinstance(d, np.integer): return int(d)
    if isinstance(d, (np.bool_, bool)): return bool(d)
    if isinstance(d, np.ndarray): return d.tolist()
    return d

meta = {
    'protocol': 'HR-aware rotation-gated 물리 decoder (최종 모델)',
    'config': dict(epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
                   sched=[SCHED_STEP, SCHED_GAMMA], n_folds=N_FOLDS, seeds=SEEDS),
    'HR_per_seed': deep_safe({s: HR_per_seed.get(s) for s in HR_per_seed}),
    'HR_3seed': deep_safe(HR_rot),
    'submission_path': sub_path,
}
with open('results/rotgate_e38_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('results/rotgate_e38_meta.json 저장')

try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    results_drive = os.path.join(DRIVE_BASE, 'results')
    os.makedirs(results_drive, exist_ok=True)
    !cp -r results/rotgate_e38_* {results_drive}/
    !cp results/rotgate_e38_meta.json {results_drive}/
    !cp {sub_path} {DRIVE_BASE}/
    files.download(sub_path)
    files.download('results/rotgate_e38_meta.json')
    print('Drive 복사 + 다운로드 완료')
except ImportError:
    print('Colab 환경 아님 — Drive 복사 skip')